### QLoRA Implementation 

In [ ]:
# =========================
# QLoRA end-to-end in ONE cell (Colab / VS Code Notebook)
# =========================
# What this cell does:
# 1) Install required libraries
# 2) Create a tiny Neo4j/Cypher dataset (instruction -> Cypher)
# 3) Tokenize the dataset (text -> input_ids, attention_mask)
# 4) Load the BASE model in 4-bit (this is the "Q" in QLoRA)
# 5) Attach LoRA adapters (this is the "LoRA" part)
# 6) Train only the adapters using Trainer
# 7) Save the adapter
# 8) Reload base(4-bit) + adapter and test generation

!pip -q install -U transformers datasets peft accelerate bitsandbytes

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

# -------------------------
# 1) Tiny training data (instruction -> Cypher)
# -------------------------
data = [
    {"instruction": "Output ONLY Cypher. How many actors are there?",
     "output": "MATCH (a:Person)-[:ACTED_IN]->(:Movie) RETURN count(DISTINCT a);"},
    {"instruction": "Output ONLY Cypher. Which actors played in the movie Casino?",
     "output": "MATCH (m:Movie {title:'Casino'})<-[:ACTED_IN]-(a:Person) RETURN a.name;"},
    {"instruction": "Output ONLY Cypher. How many movies has Tom Hanks acted in?",
     "output": "MATCH (a:Person {name:'Tom Hanks'})-[:ACTED_IN]->(m:Movie) RETURN count(m);"},
    {"instruction": "Output ONLY Cypher. Return all movie titles after 2000.",
     "output": "MATCH (m:Movie) WHERE m.released > 2000 RETURN m.title;"},
]
ds = Dataset.from_list(data)

# -------------------------
# 2) Choose a small model (for learning).
# NOTE: GPT2-style models are NOT great at instructions, but this is the simplest demo.
# -------------------------
model_id = "distilgpt2"

# -------------------------
# 3) Tokenizer: converts text -> token ids
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
# GPT2 models usually have no pad token, so we set pad = eos (safe for training)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_row(row):
    # We train the model to continue after "Cypher:" with ONLY Cypher.
    # This makes the model less likely to write English explanations.
    text = f"Instruction: {row['instruction']}\nCypher: {row['output']}"
    return {"text": text}

ds_formatted = ds.map(format_row)

max_len = 256
def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=max_len)

# This creates a new dataset with only numeric fields (input_ids, attention_mask)
tokenized = ds_formatted.map(tokenize_fn, batched=True, remove_columns=ds_formatted.column_names)

# -------------------------
# 4) QLoRA: Load base model in 4-bit using BitsAndBytesConfig
# -------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                  # <-- QLoRA switch (quantize base model to 4-bit)
    bnb_4bit_quant_type="nf4",           # best quality 4-bit type
    bnb_4bit_use_double_quant=True,      # helps quality further
    bnb_4bit_compute_dtype=torch.float16 # compute in fp16
)

# device_map="auto" places model on GPU automatically (required for bitsandbytes 4-bit)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# -------------------------
# 5) Attach LoRA adapters (trainable) on top of the frozen 4-bit base model
# target_modules depends on model architecture:
# - For GPT2/distilgpt2 it's commonly "c_attn"
# -------------------------
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                 # rank: adapter capacity (8 is a common sweet spot)
    lora_alpha=16,       # scaling strength (often 2x rank)
    lora_dropout=0.05,   # helps avoid overfitting on small datasets
    target_modules=["c_attn"],
    bias="none"
)

model = get_peft_model(model, lora_config)

# Quick confirmation: only LoRA params should be trainable
model.print_trainable_parameters()

# -------------------------
# 6) Training setup
# DataCollator pads sequences + creates labels for causal LM training
# mlm=False means "predict next token" (GPT style), not masked LM (BERT style)
# -------------------------
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="qlora_demo_out",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,  # effective batch = 2*8 = 16 (helps stability)
    num_train_epochs=10,            # tiny data needs more repeats
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=data_collator,
)

trainer.train()

# -------------------------
# 7) Save ONLY the adapter (this will create adapter_config.json + adapter_model.safetensors)
# -------------------------
adapter_dir = "qlora_adapter_v1"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

print("\nSaved adapter files:", adapter_dir)

# -------------------------
# 8) Reload base(4-bit) + adapter and TEST generation
# -------------------------
base = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
ft = PeftModel.from_pretrained(base, adapter_dir)
ft.eval()

test_prompt = "Instruction: Output ONLY Cypher. Which actors played in the movie Casino?\nCypher:"
inputs = tokenizer(test_prompt, return_tensors="pt")

# This line ensures your input tensors are moved to the same device (CPU/GPU) 
# as the model so PyTorch can run inference without errors
inputs = {k: v.to(ft.device) for k, v in inputs.items()} 

with torch.no_grad():
    out = ft.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False,             # deterministic (good for Cypher)
        repetition_penalty=1.2,      # reduce repeating sentences
        no_repeat_ngram_size=3       # avoid repeating phrases
    )

print("\n--- MODEL OUTPUT ---")
print(tokenizer.decode(out[0], skip_special_tokens=True))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 8.3 MB/s eta 0:00:00:00:0100:01


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

trainable params: 147,456 || all params: 82,060,032 || trainable%: 0.1797


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,5.702800
2,5.671800
3,5.719700
4,5.637100
5,5.674600
6,5.705000



Saved adapter files: qlora_adapter_v1


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



--- MODEL OUTPUT ---
Instruction: Output ONLY Cypher. Which actors played in the movie Casino?
Cypher: I think it's a good idea to have some of those characters that are playing, but they're not really as big or strong and so much more powerful than you'd expect from an action film like this one (I'm sure there will be other people who play) because if we had them on
